# Product Of `(1-P)` Along `run_markov_circuit` `raster_y`

This notebook studies the one-cycle linearized perturbation product

`delta G -> L delta G L^dagger`, with `L = Q_m ... Q_2 Q_1` and `Q_i = I - P_i`.

Each `P_i` is one rank-1 Wannier projector. The site order follows `run_markov_circuit(..., sequence="raster_y")`: scan `x` left-to-right, and for each `x`, scan `y` bottom-to-top.

For the default non-postselected `run_markov_circuit`, the within-site measurement order in `markov_meas_feedback` is `Ap, Am, Bp, Bm`. That is the default used below. For postselection, the project code uses `Ap, Bp, Am, Bm`; change `WITHIN_SITE_ORDER` if that is the sequence you want to compare.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from fgtn.classA_U1FGTN import classA_U1FGTN

np.set_printoptions(precision=4, suppress=True)
print(f"repo root: {REPO_ROOT}")

## Parameters

The default lattice below is intentionally modest so the full product matrix and its singular values are quick to compute. Increase `NX, NY` if you want a larger run.

In [ ]:
NX = 12
NY = 12

# Default run_markov_circuit(postselect=False) order inside markov_meas_feedback.
WITHIN_SITE_ORDER = ("Ap", "Am", "Bp", "Bm")

# Postselection order in post_selection_markov_top_layer, if needed instead:
# WITHIN_SITE_ORDER = ("Ap", "Bp", "Am", "Bm")

CONFIGS = [
    dict(label="nshNone_DW0_alpha3", nshell=None, DW=False, alpha_1=3, alpha_2=3, dw_truncation=False),
    dict(label="nsh2_DW0_alpha3", nshell=2, DW=False, alpha_1=3, alpha_2=3, dw_truncation=False),
    dict(label="nsh2_DW1_trunc_alpha1_1_alpha2_30", nshell=2, DW=True, alpha_1=1, alpha_2=30, dw_truncation=True),
    dict(label="nsh1_DW1_trunc_alpha1_1_alpha2_30", nshell=1, DW=True, alpha_1=1, alpha_2=30, dw_truncation=True),
]

print(f"N=({NX},{NY}), Nlayer={2*NX*NY}, projectors per cycle={4*NX*NY}")
print(f"within-site order: {WITHIN_SITE_ORDER}")

## Product Builder

The product is built with rank-1 updates instead of forming every dense `Q_i`:

`L <- (I - |chi><chi|) L = L - |chi>(<chi|L)`.

In [ ]:
def get_wf(model, name):
    return getattr(model, f"WF_{name}")


def normalized_chi(WF, Rx, Ry):
    chi = np.array(WF[:, Rx, Ry], dtype=np.complex128, copy=True)
    nrm = np.linalg.norm(chi)
    if not np.isfinite(nrm) or nrm == 0:
        raise ValueError(f"bad Wannier vector norm at ({Rx},{Ry}): {nrm}")
    return chi / nrm


def apply_left_Q(L, chi):
    return L - np.outer(chi, chi.conj() @ L)


def build_model_and_projectors(cfg):
    model = classA_U1FGTN(
        NX,
        NY,
        DW=cfg["DW"],
        nshell=cfg["nshell"],
        alpha_1=cfg["alpha_1"],
        alpha_2=cfg["alpha_2"],
    )
    # run_markov_circuit does not expose dw_truncation, so prebuilding is required
    # when comparing against a desired truncated-DW projector set.
    model.construct_OW_projectors(
        nshell=cfg["nshell"],
        DW=cfg["DW"],
        dw_truncation=cfg["dw_truncation"],
    )
    return model


def raster_y_coords(model):
    seq_info = model._sequence_helper("raster_y", Nx=model.Nx, Ny=model.Ny, rng=np.random.default_rng(0))
    return list(seq_info["iter_fn"]())


def q_product_for_model(model, within_site_order=WITHIN_SITE_ORDER):
    Nlayer = model.Ntot // 2
    L = np.eye(Nlayer, dtype=np.complex128)
    coords = raster_y_coords(model)
    fro_progress = []
    wf_by_name = {name: get_wf(model, name) for name in ("Ap", "Am", "Bp", "Bm")}

    step = 0
    for site_idx, (Rx, Ry) in enumerate(coords):
        for name in within_site_order:
            chi = normalized_chi(wf_by_name[name], Rx, Ry)
            L = apply_left_Q(L, chi)
            step += 1
        fro_progress.append(np.linalg.norm(L, ord="fro") / np.sqrt(Nlayer))

    return L, coords, np.array(fro_progress)


def summarize_product(L):
    svals = np.linalg.svd(L, compute_uv=False)
    eigvals = np.linalg.eigvals(L)
    tol = 1e-10
    return {
        "svals": svals,
        "eigvals": eigvals,
        "max_sv": float(svals[0]),
        "median_sv": float(np.median(svals)),
        "min_sv": float(svals[-1]),
        "rank_tol_1e-10": int(np.sum(svals > tol)),
        "nullity_tol_1e-10": int(np.sum(svals <= tol)),
        "fro_over_sqrtN": float(np.linalg.norm(L, ord="fro") / np.sqrt(L.shape[0])),
        "log10_abs_det_floor": float(np.sum(np.log10(np.maximum(svals, 1e-300)))),
    }


def print_summary_table(rows):
    header = (
        "label                                      max_sv       median_sv    min_sv       "
        "rank>1e-10  nullity  fro/sqrtN"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['label']:<42} "
            f"{row['max_sv']:10.3e}  {row['median_sv']:10.3e}  {row['min_sv']:10.3e}  "
            f"{row['rank_tol_1e-10']:10d}  {row['nullity_tol_1e-10']:7d}  {row['fro_over_sqrtN']:10.3e}"
        )

## Build Products

In [ ]:
results = {}
summary_rows = []

for cfg in CONFIGS:
    print(f"\n--- {cfg['label']} ---")
    model = build_model_and_projectors(cfg)
    L, coords, fro_progress = q_product_for_model(model)
    summary = summarize_product(L)
    summary.update(cfg)
    summary["num_sites"] = len(coords)
    summary["num_projectors"] = len(coords) * len(WITHIN_SITE_ORDER)
    if getattr(model, "DW", False):
        summary["DW_loc"] = tuple(int(x) for x in model.DW_loc)
        print(f"DW_loc={summary['DW_loc']}")
    print(f"sites={summary['num_sites']}, projectors={summary['num_projectors']}")
    print(f"max singular value={summary['max_sv']:.4e}, median={summary['median_sv']:.4e}")
    results[cfg["label"]] = dict(config=cfg, model=model, L=L, coords=coords, fro_progress=fro_progress, summary=summary)
    summary_rows.append(summary)

print("\nSummary")
print_summary_table(summary_rows)

## Singular Values Of `L`

`||delta G'||_F <= ||L||_2^2 ||delta G||_F`, so the square of the largest singular value bounds one-cycle perturbation growth in Frobenius norm.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, item in results.items():
    svals = item["summary"]["svals"]
    ax.semilogy(np.arange(1, len(svals) + 1), svals, marker=".", lw=1, ms=3, label=label)
ax.set_xlabel("singular-value index")
ax.set_ylabel("singular value of L")
ax.set_title("One raster_y sweep: L = product(I-P)")
ax.grid(True, which="both", alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()

## Progress Through The Raster Sweep

This tracks `||L||_F / sqrt(Nlayer)` after each visited site. The value is not the spectral norm, but it is a cheap view of how strongly the accumulated product is contracting as the raster scan proceeds.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, item in results.items():
    ax.semilogy(np.arange(1, len(item["fro_progress"]) + 1), item["fro_progress"], lw=1.5, label=label)
ax.set_xlabel("raster_y site index")
ax.set_ylabel("||L||_F / sqrt(Nlayer)")
ax.set_title("Accumulated contraction during one sweep")
ax.grid(True, which="both", alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()

## Matrix Magnitudes

These heatmaps show `log10(|L_ij|)` with a small floor. They are useful for spotting whether the product has a localized/banded structure after the full raster sequence.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)
for ax, (label, item) in zip(axes.flat, results.items()):
    L = item["L"]
    im = ax.imshow(np.log10(np.abs(L) + 1e-16), origin="lower", cmap="viridis", vmin=-16, vmax=0)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("j")
    ax.set_ylabel("i")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="log10(|L_ij| + 1e-16)")

## Leading Singular Vectors As Lattice Weight

The right singular vector with the largest singular value is the least-contracted single-particle direction under this one-cycle product. The plots below show its weight summed over the two orbitals at each `(x,y)`.

In [ ]:
def site_weight_from_layer_vector(v, Nx=NX, Ny=NY):
    arr = v.reshape(2, Nx, Ny, order="F")
    return np.sum(np.abs(arr) ** 2, axis=0)


fig, axes = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)
for ax, (label, item) in zip(axes.flat, results.items()):
    _, svals, vh = np.linalg.svd(item["L"], full_matrices=False)
    weight = site_weight_from_layer_vector(vh[0].conj())
    im = ax.imshow(weight.T, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"{label}\nmax sv={svals[0]:.3e}", fontsize=9)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="site weight")

## Cycle-Averaged Log Singular-Value Spectra

For `C` repeated deterministic raster sweeps, the single-particle perturbation product is `L_C = L^C`. This section plots

`lambda_i(C) = log(sigma_i(L^C)) / C`

for `C = 20, 50, 100`. These are finite-cycle, per-cycle log singular values of the product. For the covariance perturbation map `delta G -> L_C delta G L_C^dagger`, the corresponding singular-direction contraction rates are doubled.

In [ ]:
CYCLE_COUNTS = (20, 50, 100)
LOG_FLOOR = 1e-300


def cycle_averaged_log_spectrum(L, cycles, floor=LOG_FLOOR):
    Lc = np.linalg.matrix_power(L, int(cycles))
    svals = np.linalg.svd(Lc, compute_uv=False)
    return np.log(np.maximum(svals, floor)) / float(cycles), svals


cycle_spectra = {}
cycle_summary_rows = []

for label, item in results.items():
    cycle_spectra[label] = {}
    print(f"\n--- {label} ---")
    for cycles in CYCLE_COUNTS:
        log_spec, svals = cycle_averaged_log_spectrum(item["L"], cycles)
        cycle_spectra[label][cycles] = {"log_spec": log_spec, "svals": svals}
        row = {
            "label": label,
            "cycles": cycles,
            "max_log_per_cycle": float(log_spec[0]),
            "median_log_per_cycle": float(np.median(log_spec)),
            "min_log_per_cycle": float(log_spec[-1]),
            "max_sv_product": float(svals[0]),
            "rank_floor": int(np.sum(svals > LOG_FLOOR)),
        }
        cycle_summary_rows.append(row)
        print(
            f"C={cycles:3d}: max log/C={row['max_log_per_cycle']: .6f}, "
            f"median log/C={row['median_log_per_cycle']: .6f}, "
            f"min log/C={row['min_log_per_cycle']: .6f}, "
            f"max sv(L^C)={row['max_sv_product']:.3e}"
        )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)
for ax, (label, spectra) in zip(axes.flat, cycle_spectra.items()):
    for cycles in CYCLE_COUNTS:
        y = spectra[cycles]["log_spec"]
        ax.plot(np.arange(1, len(y) + 1), y, lw=1.3, label=f"C={cycles}")
    ax.axhline(0.0, color="k", lw=0.8, alpha=0.4)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("singular-value index")
    ax.set_ylabel("log sigma(L^C) / C")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
fig.suptitle("Cycle-averaged log singular-value spectra", fontsize=12)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for label, spectra in cycle_spectra.items():
    top_rates = [spectra[c]["log_spec"][0] for c in CYCLE_COUNTS]
    ax.plot(CYCLE_COUNTS, top_rates, marker="o", lw=1.5, label=label)
ax.set_xlabel("cycles C")
ax.set_ylabel("largest log sigma(L^C) / C")
ax.set_title("Top per-cycle contraction rate vs cycle count")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()

In [ ]:
header = "label                                      C    max log/C    median log/C    min log/C     max sv(L^C)"
print(header)
print("-" * len(header))
for row in cycle_summary_rows:
    print(
        f"{row['label']:<42} "
        f"{row['cycles']:3d}  "
        f"{row['max_log_per_cycle']:11.6f}  "
        f"{row['median_log_per_cycle']:13.6f}  "
        f"{row['min_log_per_cycle']:11.6f}  "
        f"{row['max_sv_product']:12.3e}"
    )

## Notes

- The product shown here is one sweep. Multiple deterministic sweeps use powers/compositions of this one-cycle product if the projector set and order are unchanged.
- For the perturbation covariance, the induced one-cycle map is `delta G -> L delta G L^dagger`.
- The DW truncated cases explicitly prebuild the Wannier projectors with `dw_truncation=True`; otherwise `run_markov_circuit` would construct the default untruncated projectors when `WF_*` are absent.